In [1]:
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

print("Libraries loaded!")

Libraries loaded!


In [2]:
# Download required NLTK data
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

print("NLTK resources downloaded!")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/touhidulislamalvi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/touhidulislamalvi/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/touhidulislamalvi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/touhidulislamalvi/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/touhidulislamalvi/nltk_data...


NLTK resources downloaded!


[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


In [4]:
import os

# Set working directory to project root
os.chdir('..')
print(f"Working directory: {os.getcwd()}")
# Load labeled data from Phase 1
df_train = pd.read_csv('data/processed/train_labeled.csv')
df_val = pd.read_csv('data/processed/val_labeled.csv')
df_test = pd.read_csv('data/processed/test_labeled.csv')

print(f"Train: {len(df_train)} rows")
print(f"Validation: {len(df_val)} rows")
print(f"Test: {len(df_test)} rows")

# Preview
df_train.head(3)

Working directory: /Users/touhidulislamalvi/Desktop/Social_Media_Mood_Analyzer
Train: 43410 rows
Validation: 5426 rows
Test: 5427 rows


,text,labels,id,mood
0,My favourite food is anything I didn't have to...,[27],eebbqej,neutral
1,"Now if he does off himself, everyone will thin...",[27],ed00q6i,neutral
2,WHY THE FUCK IS BAYLESS ISOING,[2],eezlygj,angry


In [5]:
def remove_noise(text):
    """
    Remove URLs, emojis, hashtags, and special symbols.
    
    Args:
        text: raw string
    Returns:
        cleaned string
    """
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove hashtags
    text = re.sub(r'#\w+', '', text)
    
    # Remove emojis and special symbols
    text = re.sub(r'[^\w\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Test it
test_text = "WHY THE FUCK IS BAYLESS ISOING 😡 #NBA http://nba.com"
print(remove_noise(test_text))

WHY THE FUCK IS BAYLESS ISOING


In [18]:
def normalize_text(text):
    """
    Normalize informal English words to standard form.
    
    Args:
        text: string after noise removal
    Returns:
        normalized string
    """
    # Dictionary of common informal → formal mappings
    normalization_dict = {
        "u": "you",
        "ur": "your",
        "r": "are",
        "gonna": "going to",
        "wanna": "want to",
        "gotta": "got to",
        "cant": "cannot",
        "wont": "will not",
        "ok": "okay",
        "omg": "oh my god",
        "lol": "laughing",
        "wtf": "what the",
        "idk": "i do not know",
        "imo": "in my opinion",
        "tbh": "to be honest",
        "ngl": "not going to lie",
        "abt": "about",
        "bc": "because",
        "b4": "before",
        "gr8": "great",
        "luv": "love",
        "msg": "message",
        "pls": "please",
        "thx": "thanks",
        "yr": "your",
        "irl": "in real life",
        "fyi": "for your information",
        "afaik": "as far as i know"
    }
    
    # Split into words, normalize each word, rejoin
    words = text.split()
    words = [normalization_dict.get(word.lower(), word) for word in words]
    return ' '.join(words)

# Test it
test_text = "omg u r gonna regret this tbh"
print(normalize_text(test_text))

oh my god you are going to regret this to be honest


In [7]:
def lowercase_text(text):
    """
    Convert all text to lowercase.
    
    Args:
        text: normalized string
    Returns:
        lowercased string
    """
    return text.lower()

# Test it
test_text = "WHY THE FUCK IS BAYLESS ISOING"
print(lowercase_text(test_text))

why the fuck is bayless isoing


In [19]:
def remove_punctuation(text):
    """
    Remove punctuation from text.
    
    Args:
        text: lowercased string
    Returns:
        string without punctuation
    """
    # Keep only letters, numbers and spaces
    text = re.sub(r'[^\w\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Test it
test_text = "why the fuck is bayless isoing... seriously?!"
print(remove_punctuation(test_text))

why the fuck is bayless isoing seriously


In [9]:
def tokenize_text(text):
    """
    Split text into individual word tokens.
    
    Args:
        text: cleaned string
    Returns:
        list of word tokens
    """
    return word_tokenize(text)

# Test it
test_text = "why the fuck is bayless isoing seriously"
print(tokenize_text(test_text))

['why', 'the', 'fuck', 'is', 'bayless', 'isoing', 'seriously']


In [20]:
def remove_stopwords(tokens):
    """
    Remove common English stopwords from token list.
    
    Args:
        tokens: list of word tokens
    Returns:
        list of tokens with stopwords removed
    """
    stop_words = set(stopwords.words('english'))
    return [word for word in tokens if word not in stop_words]

# Test it
test_tokens = ['why', 'the', 'fuck', 'is', 'bayless', 'isoing', 'seriously']
print(remove_stopwords(test_tokens))

['fuck', 'bayless', 'isoing', 'seriously']


In [11]:
def lemmatize_tokens(tokens):
    """
    Reduce words to their root dictionary form.
    
    Args:
        tokens: list of tokens after stopword removal
    Returns:
        list of lemmatized tokens
    """
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(word) for word in tokens]

# Test it
test_tokens = ['fuck', 'bayless', 'isoing', 'seriously', 'crying', 'happiest']
print(lemmatize_tokens(test_tokens))

['fuck', 'bayless', 'isoing', 'seriously', 'cry', 'happiest']


In [21]:
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/touhidulislamalvi/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [22]:
from nltk import pos_tag
from nltk.corpus import wordnet

def get_wordnet_pos(tag):
    """Convert NLTK POS tag to WordNet POS tag."""
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def lemmatize_tokens(tokens):
    """
    Reduce words to root form using POS aware lemmatization.
    
    Args:
        tokens: list of tokens
    Returns:
        list of lemmatized tokens
    """
    lemmatizer = WordNetLemmatizer()
    
    # Get POS tag for each token
    pos_tags = pos_tag(tokens)
    
    # Lemmatize using correct POS
    return [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) 
            for word, tag in pos_tags]

# Test it
test_tokens = ['fuck', 'bayless', 'isoing', 'seriously', 'crying', 'happiest']
print(lemmatize_tokens(test_tokens))

['fuck', 'bayless', 'isoing', 'seriously', 'cry', 'happy']


In [15]:
def preprocess(text):
    """
    Full preprocessing pipeline.
    
    Args:
        text: raw social media text
    Returns:
        cleaned preprocessed string
    """
    text = remove_noise(text)
    text = normalize_text(text)
    text = lowercase_text(text)
    text = remove_punctuation(text)
    tokens = tokenize_text(text)
    tokens = remove_stopwords(tokens)
    tokens = lemmatize_tokens(tokens)
    
    # Join tokens back into string for TF-IDF
    return ' '.join(tokens)

# Test full pipeline
test_text = "WHY THE FUCK IS BAYLESS ISOING 😡 #NBA http://nba.com"
print(preprocess(test_text))

fuck bayless isoing


In [23]:
# Apply preprocessing to all splits
print("Preprocessing train set...")
df_train['cleaned_text'] = df_train['text'].apply(preprocess)

print("Preprocessing validation set...")
df_val['cleaned_text'] = df_val['text'].apply(preprocess)

print("Preprocessing test set...")
df_test['cleaned_text'] = df_test['text'].apply(preprocess)

print("Done!")

Preprocessing train set...
Preprocessing validation set...
Preprocessing test set...
Done!


In [24]:
# Compare raw vs cleaned text
df_train[['text', 'cleaned_text', 'mood']].head(10)

,text,cleaned_text,mood
0,My favourite food is anything I didn't have to...,favourite food anything didnt cook,neutral
1,"Now if he does off himself, everyone will thin...",everyone think he laugh screw people instead a...,neutral
2,WHY THE FUCK IS BAYLESS ISOING,fuck bayless isoing,angry
3,To make her feel threatened,make feel threaten,sad
4,Dirty Southern Wankers,dirty southern wanker,angry
5,OmG pEyToN iSn'T gOoD eNoUgH tO hElP uS iN tHe...,oh god peyton isnt good enough help u playoffs...,neutral
6,Yes I heard abt the f bombs! That has to be wh...,yes hear f bomb thanks reply hubby anxiously wait,happy
7,We need more boards and to create a bit more s...,need board create bit space name well good,happy
8,Damn youtube and outrage drama is super lucrat...,damn youtube outrage drama super lucrative reddit,happy
9,It might be linked to the trust factor of your...,might link trust factor friend,neutral


In [25]:
# Save cleaned datasets
df_train.to_csv('data/processed/train_cleaned.csv', index=False)
df_val.to_csv('data/processed/val_cleaned.csv', index=False)
df_test.to_csv('data/processed/test_cleaned.csv', index=False)

print("Cleaned datasets saved!")
print(f"Train: {len(df_train)} rows")
print(f"Validation: {len(df_val)} rows")
print(f"Test: {len(df_test)} rows")

Cleaned datasets saved!
Train: 43410 rows
Validation: 5426 rows
Test: 5427 rows
